In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sys
import os
from sklearn.preprocessing import LabelEncoder


project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.nbaPlayerLogs import NBAGameLogs
from src.features.feature_engineer.pbp_features import *
from src.features.feature_engineer.pbp_min_features import *

pd.set_option('display.max_columns', None)

In [2]:
s22 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S22.csv')
s23 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S23.csv')
s24 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S24.csv')
s25 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S25.csv')
s26 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S26.csv')
p26 = pd.read_parquet(project_root / 'data/raw/pbp_stats/p26_pbp.parquet')
p26.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0042500131,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (1:10 PM EST),period,start,1,0,1
1,0042500131,4,PT12M00.00S,1,1610612739,CLE,1628386,Allen,J. Allen,0,0,0,,0,,,0,h,Jump Ball Allen vs. Poeltl: Tip to Harden,Jump Ball,,1,0,2
2,0042500131,7,PT11M45.00S,1,1610612739,CLE,1630596,Mobley,E. Mobley,0,0,0,,0,,,0,h,Mobley Out of Bounds - Bad Pass Turnover Turno...,Turnover,Out of Bounds - Bad Pass Turnover,1,0,3
3,0042500131,8,PT11M30.00S,1,1610612761,TOR,1627742,Ingram,B. Ingram,-53,168,18,Missed,1,,,0,v,MISS Ingram 18' Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,2,4
4,0042500131,9,PT11M27.00S,1,1610612739,CLE,1630596,Mobley,E. Mobley,0,0,0,,0,,,0,h,Mobley REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5


In [5]:
game_ids = p26['gameId'].unique()

res = []
for game_id in game_ids:
    print(f"Analyzing game: {game_id}")
    game_df = p26[p26['gameId'] == game_id].copy()
    game_df = cleanPlaybyPlay(game_df)
    # game_df = add_lineup_columns(game_df)
    # game_df = analyze_player_minutes(game_df)
    res.append(game_df)
final_df = pd.concat(res, ignore_index=True)
final_df.sample(5)

Analyzing game: 0042500131
Analyzing game: 0042500121
Analyzing game: 0042500161
Analyzing game: 0042500111
Analyzing game: 0042500171
Analyzing game: 0042500151
Analyzing game: 0042500101
Analyzing game: 0042500141
Analyzing game: 0042500162
Analyzing game: 0042500122
Analyzing game: 0042500152
Analyzing game: 0042500172
Analyzing game: 0042500112
Analyzing game: 0042500142
Analyzing game: 0042500132
Analyzing game: 0042500102
Analyzing game: 0042500163
Analyzing game: 0042500123
Analyzing game: 0042500133
Analyzing game: 0042500153
Analyzing game: 0042500113
Analyzing game: 0042500173
Analyzing game: 0042500143
Analyzing game: 0042500124
Analyzing game: 0042500164
Analyzing game: 0042500134
Analyzing game: 0042500103
Analyzing game: 0042500174
Analyzing game: 0042500114
Analyzing game: 0042500154
Analyzing game: 0042500144
Analyzing game: 0042500165
Analyzing game: 0042500104
Analyzing game: 0042500125
Analyzing game: 0042500155
Analyzing game: 0042500115
Analyzing game: 0042500135
A

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,MIN_SECONDS
18787,0042500135,707,PT00M05.70S,4,1610612761,TOR,1642419,Battle,J. Battle,0,0,0,,0,125,120,0,v,Battle REBOUND (Off:3 Def:3),Rebound,Unknown,1,0,502,5.7
22087,0042500136,466,PT05M02.00S,3,1610612761,TOR,1629628,Barrett,R. Barrett,10,13,2,Made,1,80,65,145,h,Barrett 2' Running Layup (20 PTS),Made Shot,Running Layup Shot,1,2,330,302.0
1488,0042500111,14,PT11M24.00S,1,1610612755,PHI,1642845,Edgecombe,V. Edgecombe,-227,17,0,Missed,1,1,0,0,v,MISS Edgecombe 3PT Jump Shot,Missed Shot,Jump Shot,1,3,9,684.0
19135,0042500175,464,PT00M01.10S,3,1610612747,LAL,1631222,LaRavia,J. LaRavia,0,0,0,,0,67,76,0,h,LaRavia BLOCK (1 BLK),,,1,2,335,1.1
20739,0042500116,627,PT00M59.80S,4,1610612738,BOS,1630625,Banton,D. Banton,-16,-2,2,Made,1,106,91,197,v,Banton 2' Driving Finger Roll Layup (2 PTS) (S...,Made Shot,Driving Finger Roll Layup Shot,1,2,455,59.8


In [13]:
final_df.to_parquet("p26_analysis.parquet", compression="snappy")

In [11]:
s26_df = pd.read_csv(project_root / 'data/raw/playoff_stats/P26.csv')
s26_df.head()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,name,POS,AGE,IS_PLAYOFF,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TEAM_SPREAD,GAME_TOTAL
0,2025-26,1628973,Jalen Brunson,Jalen,1610612752,NYK,New York Knicks,42500405,2026-06-13,NYK @ SAS,W,41.123333,14,27,0.519,4,7,0.571,13,15,0.867,1,2,3,3,3,2,0,0,1,9,45,10,56.1,0,0,59.0,1,41:07,1,94.9,107.7,107.7,94.0,96.1,96.1,0.8,11.6,11.6,0.214,1.0,7.7,0.019,0.044,0.031,7.7,7.6,0.593,0.670,0.360,0.368,97.63,90.46,75.38,90.46,0.309,78,14.0,27.0,G,4.24,3.16,3,6,9,123,0,0,79,4,8,0.50,10,19,0.526,0,0,0.000,31,87,0.356,12,37,0.324,20,28,0.714,13,35,48,14,14.0,8,4,7,21,22,94,4.0,93.7,103.3,96.4,100.0,-2.7,3.3,0.452,1.00,10.9,0.407,0.661,0.53,0.154,0.425,0.473,96.8,90.5,75.42,91,0.492,1610612759,SAS,San Antonio Spurs,33,86,0.384,12,37,0.324,12,19,0.632,14,33,47,18,13.0,6,7,4,22,21,90,-4.0,96.4,100.0,93.7,103.3,2.7,-3.3,0.545,1.38,14.2,0.339,0.593,0.47,0.144,0.453,0.477,96.8,90.5,75.42,90,0.508,Jalen Brunson,PG,29.0,1,1,1.094269,0.072951,0.072951,0,2,5.5,215.5
1,2025-26,1641705,Victor Wembanyama,Victor,1610612759,SAS,San Antonio Spurs,42500405,2026-06-13,SAS vs. NYK,L,37.866667,7,19,0.368,1,6,0.167,4,5,0.800,6,8,14,2,2,0,5,0,3,7,19,-3,51.8,1,0,46.0,1,37:52,1,104.7,105.6,105.6,100.6,106.8,106.8,4.2,-1.2,-1.2,0.100,1.0,8.0,0.146,0.182,0.165,8.0,7.9,0.395,0.448,0.274,0.284,94.54,91.27,76.06,91.27,0.151,71,7.0,19.0,C,3.99,2.64,10,12,20,61,0,0,36,5,10,0.50,2,8,0.250,2,5,0.400,33,86,0.384,12,37,0.324,12,19,0.632,14,33,47,18,13.0,6,7,4,22,21,90,-4.0,96.4,100.0,93.7,103.3,2.7,-3.3,0.545,1.38,14.2,0.339,0.593,0.47,0.144,0.453,0.477,96.8,90.5,75.42,90,0.508,1610612752,NYK,New York Knicks,31,87,0.356,12,37,0.324,20,28,0.714,13,35,48,14,14.0,8,4,7,21,22,94,4.0,93.7,103.3,96.4,100.0,-2.7,3.3,0.452,1.00,10.9,0.407,0.661,0.53,0.154,0.425,0.473,96.8,90.5,75.42,91,0.492,Victor Wembanyama,C,22.0,1,1,0.501761,0.052817,0.369718,1,0,-5.5,215.5
2,2025-26,1642844,Dylan Harper,Dylan,1610612759,SAS,San Antonio Spurs,42500405,2026-06-13,SAS vs. NYK,L,31.116667,10,19,0.526,2,4,0.500,3,6,0.500,2,3,5,4,0,0,1,1,5,4,25,-12,40.0,0,0,38.0,1,31:07,1,84.9,88.3,88.3,94.2,103.2,103.2,-9.3,-14.8,-14.8,0.364,0.0,15.4,0.049,0.075,0.062,0.0,0.0,0.579,0.578,0.289,0.295,101.35,94.87,79.06,94.87,0.248,60,10.0,19.0,NaN,4.14,2.42,4,7,10,53,0,0,31,5,10,0.50,5,9,0.556,0,0,0.000,33,86,0.384,12,37,0.324,

In [ ]:
final_df.rename(columns={"personId": "PLAYER_ID","gameId": "GAME_ID"}, inplace=True)
s26_df = s26_df.merge(final_df, on=["GAME_ID", "PLAYER_ID"], how="left")
s26_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,playerName,teamId,total_game_min,total_stints,avg_stint_min,max_stint_min,periods_played,periods_started,first_period_on,avg_entry_sec,entry_regularity_std
0,0,0,0,25893,2022-23,1629680,Matisse Thybulle,Matisse,1610612755,PHI,Philadelphia 76ers,22200001,2022-10-18,PHI @ BOS,L,0.403333,0,0,0.000,0,0,0.000,0,0,0.000,0,0,0,0,0,0,0,0,0,0,0,-1,0.0,0,0,0.0,1,0:24,1,0.0,0.0,0.0,53.2,50.0,50.0,-53.2,-50.0,-50.0,0.000,0.00,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,111.87,178.51,148.76,178.51,0.000,1,0.0,0.0,NaN,4.53,0.03,0,0,0,0,0,0,0,0,0,0.000,0,0,0.000,0,0,0.0,40,80,0.500,13,34,0.382,24,28,0.857,4,27,31,16,14.0,8,3,3,25,24,117,-9.0,114.3,119.4,126.9,129.9,-12.5,-10.5,0.400,1.14,13.1,0.190,0.744,0.457,0.143,0.581,0.634,100.8,97.5,81.25,98,0.434,1610612738,BOS,Boston Celtics,46,82,0.561,12,35,0.343,22,28,0.786,6,30,36,24,11.0,8,3,3,24,25,126,9.0,126.9,129.9,114.3,119.4,12.5,10.5,0.522,2.18,18.5,0.256,0.810,0.543,0.113,0.634,0.668,100.8,97.5,81.25,97,0.566,0,SG,25.0,3.0,216.0,3.0,216.5,0,0.000000,0.000000,0.000000,0,4,M. Thybulle,1.610613e+09,0.40,2.0,0.20,0.34,2.0,0.0,1.0,12.70,12.45
1,25,1,25,25867,2022-23,203210,JaMychal Green,JaMychal,1610612744,GSW,Golden State Warriors,22200002,2022-10-18,GSW vs. LAL,W,23.433333,3,6,0.500,2,3,0.667,0,0,0.000,5,2,7,0,0,1,0,0,1,0,8,-3,19.4,0,0,19.0,1,23:26,1,89.3,91.1,91.1,97.5,98.2,98.2,-8.1,-7.1,-7.1,0.000,0.00,0.0,0.152,0.077,0.119,0.0,0.0,0.667,0.667,0.087,0.091,115.20,113.68,94.74,113.68,0.117,56,3.0,6.0,NaN,4.13,1.63,5,5,10,29,0,0,23,1,3,0.333,2,3,0.667,2,4,0.5,45,99,0.455,16,45,0.356,17,23,0.739,11,37,48,31,18.0,11,4,4,23,18,123,14.0,105.9,107.0,92.4,97.3,13.6,9.6,0.689,1.72,19.1,0.281,0.754,0.518,0.157,0.535,0.564,117.1,113.5,94.58,115,0.548,1610612747,LAL,Los Angeles Lakers,40,94,0.426,10,40,0.250,19,25,0.760,9,39,48,23,22.0,12,4,4,18,23,109,-14.0,92.4,97.3,105.9,107.0,-13.6,-9.6,0.575,1.05,15.4,0.246,0.719,0.482,0.196,0.479,0.519,117.1,113.5,94.58,112,0.452,0,PF,32.0,-7.5,223.5,-7.5,223.5,0,0.341394,0.000000,0.298720,1,1,J. Green,1.610613e+09,23.43,4.0,5.86,10.13,4.0,2.0,1.0,535.00,213.78
2,26,2,26,25866,2022-23,1629684,Gra

In [27]:
s23_df.to_csv(project_root / 'data/raw/season_stats/s23_pbp.csv', index=False)